In [3]:
import os
import hashlib
import json
import ast
import difflib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import PyPDF2
import docx
import nbformat

# =========================================================
# CONFIG
# =========================================================
BASE_DIR = "/Users/mohammadafsheen/Desktop/lms_parent_portal/evalsea-duplicate-detection-engine/sample_assignments"  # ✅ CHANGE IF NEEDED
SIMILARITY_THRESHOLD = 0.75   # Only report above this
FAST_SIGNATURE_WORDS = 180    # Controls bucket tightness

# =========================================================
# FILE EXTRACTION
# =========================================================
def extract_text(path):
    if path.endswith(".pdf"):
        text = ""
        reader = PyPDF2.PdfReader(open(path, "rb"))
        for p in reader.pages:
            t = p.extract_text()
            if t:
                text += t + "\n"
        return text.strip()

    if path.endswith(".docx"):
        d = docx.Document(path)
        return "\n".join([p.text for p in d.paragraphs])

    if path.endswith(".ipynb"):
        nb = nbformat.read(path, as_version=4)
        return "\n".join(cell.source for cell in nb.cells)

    if path.endswith(".py") or path.endswith(".txt"):
        return open(path, "r", errors="ignore").read()

    return ""

# =========================================================
# FAST SIGNATURE (CANDIDATE FILTER)
# =========================================================
def fast_signature(text):
    tokens = text.lower().split()[:FAST_SIGNATURE_WORDS]
    base = " ".join(tokens)
    return hashlib.md5(base.encode()).hexdigest()

# =========================================================
# AST STRUCTURE SIMILARITY (FOR PYTHON)
# =========================================================
def ast_similarity(code1, code2):
    try:
        t1 = ast.dump(ast.parse(code1))
        t2 = ast.dump(ast.parse(code2))
        return difflib.SequenceMatcher(None, t1, t2).ratio()
    except:
        return 0.0

# =========================================================
# LOAD LOCAL SUBMISSIONS
# =========================================================
def load_submissions():
    print("🔍 Loading local submissions...")

    submissions = {}

    for student in os.listdir(BASE_DIR):
        student_path = os.path.join(BASE_DIR, student)

        if not os.path.isdir(student_path):
            continue

        for file in os.listdir(student_path):
            full_path = os.path.join(student_path, file)

            submissions[student] = {
                "file": file,
                "path": full_path,
                "text": extract_text(full_path),
                "is_code": file.endswith(".py")
            }
            break

    print("✅ Total submissions found:", len(submissions))
    return submissions

# =========================================================
# BUCKETING (LIKELY MATCH GROUPS)
# =========================================================
def build_candidate_buckets(submissions):
    print("⚡ Building fast candidate buckets...")
    buckets = {}

    for sid, data in submissions.items():
        sig = fast_signature(data["text"])
        buckets.setdefault(sig, []).append(sid)

    print("✅ Total buckets formed:", len(buckets))
    return buckets

# =========================================================
# SEMANTIC SIMILARITY (TF-IDF)
# =========================================================
def compute_semantic_matrix(texts):
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf = vectorizer.fit_transform(texts)
    return cosine_similarity(tfidf)

# =========================================================
# MAIN OPTIMIZED DETECTOR
# =========================================================
def run_optimized_detector():
    submissions = load_submissions()
    buckets = build_candidate_buckets(submissions)

    final_results = []

    print("🤖 Running deep similarity ONLY inside candidate buckets...\n")

    for bucket_key, student_list in buckets.items():

        # ✅ Skip single-file buckets instantly
        if len(student_list) < 2:
            continue

        texts = [submissions[s]["text"] for s in student_list]
        sim_matrix = compute_semantic_matrix(texts)

        # ✅ Compare ONLY inside this small bucket
        for i in range(len(student_list)):
            for j in range(i + 1, len(student_list)):
                score = sim_matrix[i][j]

                if score >= SIMILARITY_THRESHOLD:
                    s1 = student_list[i]
                    s2 = student_list[j]

                    ast_score = 0.0
                    if submissions[s1]["is_code"] and submissions[s2]["is_code"]:
                        ast_score = ast_similarity(
                            submissions[s1]["text"],
                            submissions[s2]["text"]
                        )

                    final_results.append({
                        "student_1": s1,
                        "student_2": s2,
                        "file_1": submissions[s1]["file"],
                        "file_2": submissions[s2]["file"],
                        "semantic_similarity": round(float(score), 4),
                        "ast_similarity": round(float(ast_score), 4),
                        "verdict": "DUPLICATE"
                    })

    print("\n✅ FINAL OPTIMIZED DUPLICATE REPORT\n")
    print(json.dumps(final_results, indent=2))

# =========================================================
# ENTRY POINT
# =========================================================
def lambda_handler(event, context):
    """
    ✅ LOCAL TEST EVENT FORMAT:
    {
        "assignment_path": "/Users/mohammadafsheen/Desktop/sample_assignments",
        "similarity_threshold": 0.75   (optional)
    }
    """

    try:
        # -----------------------------
        # STEP 1: READ EVENT INPUT
        # -----------------------------
        assignment_path = event.get("assignment_path")
        SIMILARITY_THRESHOLD = float(event.get("similarity_threshold", 0.75))

        if not assignment_path or not os.path.exists(assignment_path):
            return {
                "statusCode": 400,
                "body": json.dumps({"error": "Valid assignment_path not provided"})
            }

        # -----------------------------
        # STEP 2: LOAD SUBMISSIONS
        # -----------------------------
        submissions = {}

        for student in os.listdir(assignment_path):
            student_path = os.path.join(assignment_path, student)
            if not os.path.isdir(student_path):
                continue

            for file in os.listdir(student_path):
                full_path = os.path.join(student_path, file)

                submissions[student] = {
                    "file": file,
                    "path": full_path,
                    "text": extract_text(full_path),
                    "is_code": file.endswith(".py")
                }
                break

        if len(submissions) < 2:
            return {
                "statusCode": 200,
                "body": json.dumps({
                    "message": "Not enough submissions to check duplicates"
                })
            }

        # -----------------------------
        # STEP 3: BUILD CANDIDATE BUCKETS
        # -----------------------------
        buckets = build_candidate_buckets(submissions)

        # -----------------------------
        # STEP 4: OPTIMIZED COMPARISON
        # -----------------------------
        final_results = []

        for bucket_key, student_list in buckets.items():
            if len(student_list) < 2:
                continue

            texts = [submissions[s]["text"] for s in student_list]
            sim_matrix = compute_semantic_matrix(texts)

            for i in range(len(student_list)):
                for j in range(i + 1, len(student_list)):
                    score = float(sim_matrix[i][j])

                    if score >= SIMILARITY_THRESHOLD:
                        s1 = student_list[i]
                        s2 = student_list[j]

                        ast_score = 0.0
                        if submissions[s1]["is_code"] and submissions[s2]["is_code"]:
                            ast_score = ast_similarity(
                                submissions[s1]["text"],
                                submissions[s2]["text"]
                            )

                        final_results.append({
                            "student_1": s1,
                            "student_2": s2,
                            "file_1": submissions[s1]["file"],
                            "file_2": submissions[s2]["file"],
                            "semantic_similarity": round(score, 4),
                            "ast_similarity": round(ast_score, 4),
                            "verdict": "DUPLICATE"
                        })

        # -----------------------------
        # STEP 5: RETURN RESPONSE
        # -----------------------------
        result = {
            "assignment_path": assignment_path,
            "total_students": len(submissions),
            "candidate_buckets": len(buckets),
            "checked_pairs": len(final_results),
            "duplicate_results": final_results
        }

        return {
            "statusCode": 200,
            "body": json.dumps(result)
        }

    except Exception as e:
        return {
            "statusCode": 500,
            "body": json.dumps({"error": str(e)})
        }


In [7]:
event={
  "assignment_path": "/Users/mohammadafsheen/Desktop/lms_parent_portal/evalsea-duplicate-detection-engine/sample_assignments",
  "similarity_threshold": 0.75
}


In [8]:
response = lambda_handler(event, None)
print(json.dumps(response, indent=2))


⚡ Building fast candidate buckets...
✅ Total buckets formed: 2
{
  "statusCode": 200,
  "body": "{\"assignment_path\": \"/Users/mohammadafsheen/Desktop/lms_parent_portal/evalsea-duplicate-detection-engine/sample_assignments\", \"total_students\": 3, \"candidate_buckets\": 2, \"checked_pairs\": 1, \"duplicate_results\": [{\"student_1\": \"student1@gmail.com\", \"student_2\": \"student2@gmail.com\", \"file_1\": \"assignment1.pdf\", \"file_2\": \"assignment2.pdf\", \"semantic_similarity\": 1.0, \"ast_similarity\": 0.0, \"verdict\": \"DUPLICATE\"}]}"
}
